# 02 — Phần 1: Câu hỏi trắc nghiệm

10 câu hỏi MCQ, mỗi câu 2 điểm. Tất cả tính toán trực tiếp từ dữ liệu.

Mỗi câu: code tính → in kết quả → đáp án + lý giải ngắn. Cuối notebook: bảng tổng hợp 10 đáp án.

## Mục lục
- [Setup](#setup)
- [Q1 — Inter-order gap median](#q1)
- [Q2 — Segment margin cao nhất](#q2)
- [Q3 — Return reason cho Streetwear](#q3)
- [Q4 — Traffic source bounce rate thấp nhất](#q4)
- [Q5 — % order_items có promo](#q5)
- [Q6 — Age group đơn/khách cao nhất](#q6)
- [Q7 — Region revenue cao nhất](#q7)
- [Q8 — Payment method trong đơn cancelled](#q8)
- [Q9 — Size có return rate cao nhất](#q9)
- [Q10 — Installments có payment_value TB cao nhất](#q10)
- [Tổng hợp đáp án](#summary)

## <a id='setup'></a>Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

DATA_DIR = Path('../data/raw')

products    = pd.read_csv(DATA_DIR / 'products.csv')
customers   = pd.read_csv(DATA_DIR / 'customers.csv')
promotions  = pd.read_csv(DATA_DIR / 'promotions.csv')
geography   = pd.read_csv(DATA_DIR / 'geography.csv')
orders      = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')
payments    = pd.read_csv(DATA_DIR / 'payments.csv')
shipments   = pd.read_csv(DATA_DIR / 'shipments.csv', parse_dates=['ship_date','delivery_date'])
returns     = pd.read_csv(DATA_DIR / 'returns.csv', parse_dates=['return_date'])
reviews     = pd.read_csv(DATA_DIR / 'reviews.csv', parse_dates=['review_date'])
sales       = pd.read_csv(DATA_DIR / 'sales.csv', parse_dates=['Date'])
web_traffic = pd.read_csv(DATA_DIR / 'web_traffic.csv', parse_dates=['date'])

print('Loaded:', {k: v.shape for k,v in {
    'products':products,'customers':customers,'promotions':promotions,'geography':geography,
    'orders':orders,'order_items':order_items,'payments':payments,'shipments':shipments,
    'returns':returns,'reviews':reviews,'sales':sales,'web_traffic':web_traffic
}.items()})

ANSWERS = {}  # tích lũy đáp án để in cuối notebook

Loaded: {'products': (2412, 8), 'customers': (121930, 7), 'promotions': (50, 10), 'geography': (39948, 4), 'orders': (646945, 8), 'order_items': (714669, 7), 'payments': (646945, 4), 'shipments': (566067, 4), 'returns': (39939, 7), 'reviews': (113551, 7), 'sales': (3833, 3), 'web_traffic': (3652, 7)}


## <a id='q1'></a>Q1 — Trung vị số ngày giữa hai lần mua liên tiếp (khách >1 đơn)

**Phép tính:** Với mỗi customer có ≥2 đơn, sort theo `order_date`, tính diff ngày giữa các đơn liên tiếp → median **toàn bộ gaps**.

In [2]:
# Lọc khách hàng có >1 đơn
order_counts = orders.groupby('customer_id').size()
multi_buyers = order_counts[order_counts > 1].index

o = orders[orders['customer_id'].isin(multi_buyers)][['customer_id','order_date']].copy()
o = o.sort_values(['customer_id','order_date'])
o['gap_days'] = o.groupby('customer_id')['order_date'].diff().dt.days

gaps = o['gap_days'].dropna()
q1 = gaps.median()

print(f'Số khách có >1 đơn: {len(multi_buyers):,}')
print(f'Tổng số gap: {len(gaps):,}')
print(f'Median inter-order gap: {q1:.1f} ngày')
print(f'\nDistribution:')
print(gaps.describe().to_string())

ANSWERS['Q1'] = f'{q1:.1f} ngày'

Số khách có >1 đơn: 67,888
Tổng số gap: 556,699
Median inter-order gap: 144.0 ngày

Distribution:
count   556,699.0000
mean        285.5925
std         389.6916
min           0.0000
25%          46.0000
50%         144.0000
75%         357.0000
max       3,785.0000


**📌 Đáp án Q1:** `{Q1}` — chọn đáp án trắc nghiệm gần nhất.

## <a id='q2'></a>Q2 — Segment có margin trung bình cao nhất

**Margin per product:** `(price − cogs) / price`. Lấy mean theo `segment`, sort desc.

In [3]:
p = products.copy()
p['margin'] = (p['price'] - p['cogs']) / p['price']

q2_table = p.groupby('segment')['margin'].agg(['mean','count']).sort_values('mean', ascending=False)
print(q2_table.to_string())

q2 = q2_table.index[0]
print(f'\n→ Segment margin cao nhất: {q2} ({q2_table.iloc[0,0]*100:.2f}%)')
ANSWERS['Q2'] = q2

              mean  count
segment                  
Standard    0.3134    262
Premium     0.2854    177
All-weather 0.2842    169
Activewear  0.2656    598
Performance 0.2636    347
Balanced    0.2580    306
Trendy      0.2408    148
Everyday    0.2363    405

→ Segment margin cao nhất: Standard (31.34%)


**📌 Đáp án Q2:** `{Q2}`

## <a id='q3'></a>Q3 — Lý do trả hàng phổ biến nhất cho Streetwear

Join `returns` ⨝ `products` để có category, filter `Streetwear`, đếm `return_reason`.

In [4]:
m = returns.merge(products[['product_id','category']], on='product_id', how='left')
sw = m[m['category']=='Streetwear']

q3_table = sw['return_reason'].value_counts()
print(f'Tổng đơn return Streetwear: {len(sw):,}')
print(f'\nReturn reason distribution:')
print(q3_table.to_string())

q3 = q3_table.index[0]
print(f'\n→ Lý do phổ biến nhất: {q3} ({q3_table.iloc[0]:,} dòng, {q3_table.iloc[0]/len(sw)*100:.1f}%)')
ANSWERS['Q3'] = q3

Tổng đơn return Streetwear: 21,799

Return reason distribution:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159

→ Lý do phổ biến nhất: wrong_size (7,626 dòng, 35.0%)


**📌 Đáp án Q3:** `{Q3}`

## <a id='q4'></a>Q4 — Traffic source có bounce_rate TB thấp nhất

In [5]:
q4_table = web_traffic.groupby('traffic_source')['bounce_rate'].agg(['mean','count']).sort_values('mean')
print(q4_table.to_string())

q4 = q4_table.index[0]
print(f'\n→ Traffic source bounce_rate thấp nhất: {q4} ({q4_table.iloc[0,0]*100:.2f}%)')
ANSWERS['Q4'] = q4

                 mean  count
traffic_source              
email_campaign 0.0045    505
social_media   0.0045    632
paid_search    0.0045    784
referral       0.0045    375
organic_search 0.0045   1090
direct         0.0045    266

→ Traffic source bounce_rate thấp nhất: email_campaign (0.45%)


**📌 Đáp án Q4:** `{Q4}`

## <a id='q5'></a>Q5 — % dòng order_items có khuyến mãi (`promo_id` không null)

⚠️ **CLAUDE.md warn:** chỉ xét `promo_id`, **không** `promo_id_2` (vốn 100% null).

In [6]:
total = len(order_items)
with_promo = order_items['promo_id'].notna().sum()
pct = with_promo / total * 100

print(f'Tổng order_items: {total:,}')
print(f'Có promo_id: {with_promo:,}')
print(f'% có promo: {pct:.2f}%')

ANSWERS['Q5'] = f'{pct:.2f}%'

Tổng order_items: 714,669
Có promo_id: 276,316
% có promo: 38.66%


**📌 Đáp án Q5:** `{Q5}`

## <a id='q6'></a>Q6 — Age group có số đơn TB / khách hàng cao nhất

**Phép tính:** `tổng_đơn / số_khách_unique` theo `age_group`. Bao gồm cả khách hàng chưa từng đặt đơn? → đề nói "số đơn TB / khách hàng" → mẫu = số khách trong age_group đó. **Bao gồm cả khách 0 đơn** (sẽ kéo trung bình xuống).

In [7]:
# Đếm số đơn / customer
order_per_cust = orders.groupby('customer_id').size().rename('n_orders')
m = customers[['customer_id','age_group']].merge(order_per_cust, on='customer_id', how='left').fillna(0)

q6_table = m.groupby('age_group').agg(
    total_orders=('n_orders','sum'),
    n_customers=('customer_id','count'),
).assign(orders_per_customer=lambda d: d['total_orders']/d['n_customers']).sort_values('orders_per_customer', ascending=False)

print(q6_table.to_string())

q6 = q6_table.index[0]
print(f'\n→ Age group đơn/khách cao nhất: {q6} ({q6_table.iloc[0,2]:.2f} đơn/khách)')
ANSWERS['Q6'] = q6

           total_orders  n_customers  orders_per_customer
age_group                                                
55+         72,760.0000        13457               5.4069
45-54      124,138.0000        23172               5.3572
35-44      170,368.0000        31920               5.3373
25-34      190,622.0000        36342               5.2452
18-24       89,057.0000        17039               5.2267

→ Age group đơn/khách cao nhất: 55+ (5.41 đơn/khách)


**📌 Đáp án Q6:** `{Q6}`

## <a id='q7'></a>Q7 — Region tạo doanh thu cao nhất

**Lưu ý đề:** "geography ⨝ sales_train". Nhưng `sales.csv` chỉ có `Date,Revenue,COGS` (không có region). → Phải tính revenue qua **`order_items` + `orders` + `geography`**.

**Revenue per order_item:** `quantity × unit_price − discount_amount`.

In [8]:
oi = order_items.copy()
oi['revenue'] = oi['quantity'] * oi['unit_price'] - oi['discount_amount']

m = oi.merge(orders[['order_id','zip','order_status']], on='order_id', how='left')
m = m.merge(geography[['zip','region']], on='zip', how='left')

# Lấy tất cả status (đề không loại cancelled). Nếu cần loại, có thể test thêm.
q7_table = m.groupby('region')['revenue'].sum().sort_values(ascending=False)
print('Revenue theo region (tất cả order_status):')
print(q7_table.apply(lambda x: f'{x:,.0f}').to_string())

# Sanity check: chỉ delivered
q7_table_delivered = m[m['order_status']=='delivered'].groupby('region')['revenue'].sum().sort_values(ascending=False)
print('\nRevenue theo region (chỉ delivered):')
print(q7_table_delivered.apply(lambda x: f'{x:,.0f}').to_string())

q7 = q7_table.index[0]
print(f'\n→ Region revenue cao nhất: {q7}')
ANSWERS['Q7'] = q7

Revenue theo region (tất cả order_status):
region
East       7,291,150,819
Central    4,719,491,268
West       3,670,227,178

Revenue theo region (chỉ delivered):
region
East       5,826,256,457
Central    3,762,669,980
West       2,929,249,520

→ Region revenue cao nhất: East


**📌 Đáp án Q7:** `{Q7}`

## <a id='q8'></a>Q8 — Payment method phổ biến nhất trong đơn `cancelled`

In [9]:
cancelled = orders[orders['order_status']=='cancelled']
q8_table = cancelled['payment_method'].value_counts()
print(f'Tổng đơn cancelled: {len(cancelled):,}')
print(q8_table.to_string())

q8 = q8_table.index[0]
print(f'\n→ Payment method phổ biến nhất trong cancelled: {q8} ({q8_table.iloc[0]:,} đơn, {q8_table.iloc[0]/len(cancelled)*100:.1f}%)')
ANSWERS['Q8'] = q8

Tổng đơn cancelled: 59,462
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535

→ Payment method phổ biến nhất trong cancelled: credit_card (28,452 đơn, 47.8%)


**📌 Đáp án Q8:** `{Q8}`

## <a id='q9'></a>Q9 — Size (S/M/L/XL) có tỷ lệ trả hàng cao nhất

**Tỷ lệ trả hàng (size):** `Σ return_quantity / Σ quantity sold` cho cùng size.

Phải merge `returns`/`order_items` ⨝ `products` để biết `size`.

In [10]:
# Số quantity sold theo size (qua order_items)
oi_size = order_items.merge(products[['product_id','size']], on='product_id', how='left')
sold = oi_size.groupby('size')['quantity'].sum()

# Số quantity returned theo size
ret_size = returns.merge(products[['product_id','size']], on='product_id', how='left')
returned = ret_size.groupby('size')['return_quantity'].sum()

q9_table = pd.DataFrame({'sold': sold, 'returned': returned})
q9_table['return_rate'] = q9_table['returned'] / q9_table['sold']
q9_table = q9_table.sort_values('return_rate', ascending=False)
print(q9_table.to_string())

q9 = q9_table.index[0]
print(f'\n→ Size có return rate cao nhất: {q9} ({q9_table.iloc[0,2]*100:.2f}%)')
ANSWERS['Q9'] = q9

        sold  returned  return_rate
size                               
S     774468     26797       0.0346
L     778599     26688       0.0343
M     792889     26932       0.0340
XL    867187     29169       0.0336

→ Size có return rate cao nhất: S (3.46%)


**📌 Đáp án Q9:** `{Q9}`

## <a id='q10'></a>Q10 — Installments có payment_value TB / đơn cao nhất

In [11]:
q10_table = payments.groupby('installments').agg(
    n_orders=('order_id','count'),
    avg_payment=('payment_value','mean'),
).sort_values('avg_payment', ascending=False)
print(q10_table.to_string())

q10 = q10_table.index[0]
print(f'\n→ Installments plan có avg payment cao nhất: {q10} kỳ ({q10_table.iloc[0,1]:,.2f})')
ANSWERS['Q10'] = f'{q10} kỳ'

              n_orders  avg_payment
installments                       
6               109910  24,446.6544
3               218949  24,399.6355
12               54126  24,245.7727
1               262866  24,113.2742
2                 1094     708.4737

→ Installments plan có avg payment cao nhất: 6 kỳ (24,446.65)


**📌 Đáp án Q10:** `{Q10}`

## <a id='summary'></a>Tổng hợp 10 đáp án

In [12]:
print('='*50)
print('   ĐÁP ÁN MCQ — PHẦN 1')
print('='*50)
for q, ans in ANSWERS.items():
    print(f'  {q}: {ans}')
print('='*50)

   ĐÁP ÁN MCQ — PHẦN 1
  Q1: 144.0 ngày
  Q2: Standard
  Q3: wrong_size
  Q4: email_campaign
  Q5: 38.66%
  Q6: 55+
  Q7: East
  Q8: credit_card
  Q9: S
  Q10: 6 kỳ
